# Language Model and Sequence Generation — Transformers

In this lab, we'll study the different existing ways to decode text given a trained model. This step is needed for many natural language processing tasks, including translation, question answering and generation.

We'll take text generation as an example: until very recently, it necessarily relied on more or less sophisticated fill-in-the-blank texts. The arrival of powerful models trained on large amounts of data changed the game: it's now possible to use neural methods.

In this notebook, we'll see how to generate text from a trained model, starting from the simplest method and working up to state-of-the-art methods. This will give you a solid understanding of the mechanisms of neural networks applied to text: in particular the use of the softmax activation, which is at the heart of the main state-of-the-art NLP architectures.

An [excellent blog post](https://huggingface.co/blog/how-to-generate) on the subject by Hugging Face can serve as a complementary resource to this notebook.

## Installation

The [`transformers`](https://github.com/huggingface/transformers) library developed by the French-American startup [Hugging Face](https://huggingface.co/) provides a large number of models of the transformer family.

It's easy to install with `pip`:

In [ ]:
!pip install 'transformers < 5'

## Choosing the Language

We'll see how to generate text in English but also in French, by loading models pre-trained on different corpora. The next cell lets you change the language used for the rest of the lab.

In [ ]:
lang = "en"
# lang = "fr"

## Importing a Trained Model

Besides providing the source code of many transformer architectures, the `transformers` library lets you download [many trained models](https://huggingface.co/transformers/pretrained_models.html).

For this lab, we'll use the GPT2 architecture, behind the [model that surprised the NLP community](https://openai.com/blog/better-language-models/) with the quality of its generations (and whose successor, GPT3, has since reached [even more impressive results](https://github.com/elyase/awesome-gpt3)).

To do so, we simply call the `from_pretrained` method on the class of the model architecture we're interested in, with the name of the training we want to download as argument. The tokenizer tied to the model can be downloaded the same way.

In [ ]:
import functools
import typing

import numpy
import tensorflow
import transformers
import tqdm.notebook


pretraining_name = "gpt2" if lang == "en" else "antoiloui/belgpt2"

model = transformers.TFGPT2LMHeadModel.from_pretrained(pretraining_name, use_safetensors=False)
tokenizer = transformers.GPT2Tokenizer.from_pretrained(pretraining_name, use_safetensors=False)

## Using the GPU

By default, TensorFlow uses the available GPUs. Let's check whether Colab gave us one:

In [4]:
print(tensorflow.config.experimental.list_physical_devices("GPU"))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## Inspecting the Model

After downloading a pre-trained model, it's always a good idea to check its architecture and characteristics.

To do so, in the `transformers` library, we can check its config.

In [5]:
print(model.config)

GPT2Config {
  "_name_or_path": "gpt2",
  "activation_function": "gelu_new",
  "architectures": [
    "GPT2LMHeadModel"
  ],
  "attn_pdrop": 0.1,
  "bos_token_id": 50256,
  "embd_pdrop": 0.1,
  "eos_token_id": 50256,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "gpt2",
  "n_ctx": 1024,
  "n_embd": 768,
  "n_head": 12,
  "n_inner": null,
  "n_layer": 12,
  "n_positions": 1024,
  "reorder_and_upcast_attn": false,
  "resid_pdrop": 0.1,
  "scale_attn_by_inverse_layer_idx": false,
  "scale_attn_weights": true,
  "summary_activation": null,
  "summary_first_dropout": 0.1,
  "summary_proj_to_labels": true,
  "summary_type": "cls_index",
  "summary_use_proj": true,
  "task_specific_params": {
    "text-generation": {
      "do_sample": true,
      "max_length": 50
    }
  },
  "transformers_version": "4.19.4",
  "use_cache": true,
  "vocab_size": 50257
}



- It has 12 layers
- Its main regularization mechanism is dropout
- Its training stabilization mechanisms visible from the architecture shown above are layer normalization and residual connections.
- We can read its value in the `bos_token_id` and `eos_token_id` keys of the config: `50256`.

## The Generation Mechanism

To generate text from a trained model, we work iteratively: we generate the words one by one, starting with an empty text or a sentence we want to complete.

To generate a word, we feed the network the words generated so far (turned into embedding indices). The network then outputs a score for each word of the vocabulary. We pick one of these words before looping (appending this word to what had been generated, then feeding the result to the network to generate the next word).

More formally, we try to maximize $P(\text{Text}| \text{Prompt})$ with the following decomposition:

$$P(\text{Text}| \text{Prompt}) = \Pi_{i=1}^{\text{Text length}} P(\text{Word}_i| \text{Word}_{j<i}, \text{Prompt})$$

The whole challenge of decoding is to find the maximal value of this product: it's the model's best proposal. This value is impossible to compute exactly, so we resort to heuristics to approximate it.

## Tokenization

The tokenizer downloaded along with the model turns an input sentence into embedding indices the model can understand.

In the rest of the lab, we'll use the following `encode` and `decode` functions, which turn strings into vocabulary indices the model understands (they're the indices of its embedding module).

In [ ]:
def encode(sentence: str) -> tensorflow.Tensor:
  tokens = tokenizer.encode(sentence,
                            add_special_tokens=False,
                            return_tensors="tf")
  bos = tensorflow.constant([[model.config.bos_token_id]], dtype="int64")
  return tensorflow.concat([bos, tensorflow.cast(tokens, "int64")], axis=1)


def decode(tokens: tensorflow.Tensor) -> str:
  if tokens.ndim == 2:
    tokens = tensorflow.squeeze(tokens)
  return tokenizer.decode(tokens[1:], clean_up_tokenization_spaces=True)

## Studying the Outputs of the Pre-Trained Model

The first step to decode sentences from a trained model is to run a forward pass of the model to get its predictions for the next word. Let's study its behavior.

In Keras, to compute the forward pass of a model, we directly use `model(input_tensor)`, as we've already seen.

Note the `past` argument: it avoids recomputing values computed during the previous decoding step. If we give the past argument to the model, we must not pass the old values again.

In [7]:
def describe_shapes(input_string: str,
                    output_tensor: tensorflow.Tensor,
                    output_past: list[tensorflow.Tensor]
                   ) -> str:
  print("-" * 80)
  print(f"Output shape for the input '{input_string}': "
        f"{output_tensor.shape}")
  print(f"Past length: {len(output_past)}")
  print(f"Past type: {type(output_past[0])}")


empty_input = ""
descartes_input = ("Je pense, donc je suis"
                   if lang == "fr"
                   else "I think, therefore I am")

empty_output, empty_past = model(encode(empty_input),
                                 return_dict=False)
descartes_output, descartes_past = model(encode(descartes_input),
                                         return_dict=False)

describe_shapes(empty_input, empty_output, empty_past)
describe_shapes(descartes_input, descartes_output, descartes_past)

--------------------------------------------------------------------------------
Output shape for the input '': (1, 1, 50257)
Past length: 12
Past type: <class 'tensorflow.python.framework.ops.EagerTensor'>
--------------------------------------------------------------------------------
Output shape for the input 'I think, therefore I am': (1, 7, 50257)
Past length: 12
Past type: <class 'tensorflow.python.framework.ops.EagerTensor'>


## A `forward` Function Suited to Decoding

Given the observations made in the previous section, we'll use the following `forward` function, which returns an output suited to decoding: only the last column interests us (the prediction of the next word). This `forward` function implements this selection.

In [ ]:
def forward(tokens: tensorflow.Tensor,
            past: list[tensorflow.Tensor] | None = None
           ) -> tuple[tensorflow.Tensor, list[tensorflow.Tensor]]:
  logits, new_past = model(tokens, past_key_values=past, return_dict=False)
  return logits[:, -1, :], new_past

Note that when the `past` argument is used, we must not give again the tokens that were already computed, only the new token to decode. So we must use either:

- `forward(all_tokens)`
- `forward(last_token, past)`

where `all_tokens` would for example be a sequence of 7 tokens of shape `(1, 7, 50257)`, whereas `last_token` would be of shape `(1, 1, 50257)`.

## Decoding Loop

A decoding loop always follows the same pattern:

1. Encode everything produced so far (inputs and previous outputs)
2. Produce the index of the most likely next word
3. Repeat 1. and 2. until a stopping criterion is met (in this lab, we'll only use the number of words produced)

Step 2 is where everything happens. For now, we'll code everything else, so we can focus on step 2 afterwards.

In [ ]:
def decoding_loop(step_function) -> str:

  @functools.wraps(step_function)
  def wrapper(prompt, length, *step_args, **step_kwargs):
    token_ids = encode(prompt)
    past = None
    decoded = [token_ids]
    for i in tqdm.notebook.trange(length, desc="Words", leave=False):
      logits, past = forward(decoded[-1], past)
      index = step_function(logits, *step_args, **step_kwargs)
      decoded += [index[None, None]]
    decoded_tensor = tensorflow.concat(decoded, axis=1)
    return decode(decoded_tensor)

  return wrapper

## Greedy Decoding

Greedy decoding is the simplest method to decode text from a trained model.

At each decoding step (to decode each word), it takes the word with the highest probability.

More formally, in the equation:

$$P(\text{Text}|\text{Prompt}) = \Pi_{i=0}^{\text{Text length}} P(\text{Word}_i| \text{Word}_{j<i}, \text{Prompt})$$

greedy decoding picks $\text{Word}_i$ to maximize $P(\text{Word}_i| \text{Word}_{j<i}, \text{Prompt})$. However, we often need to pick a less likely word to then reach very likely words.

The following example is given in Hugging Face's blog post on text generation:

![Greedy decoding](https://huggingface.co/blog/assets/02_how-to-generate/greedy_search.png)

We can see that after decoding "The", we pick "nice" because it's the word with the highest probability for the model, and that the complete decoding "The nice woman" has a lower probability (0.20) than the decoding we would have obtained by picking "dog" ("The dog has", probability 0.36).

In [ ]:
@decoding_loop
def greedy(logits: tensorflow.Tensor) -> tensorflow.Tensor:
  return tensorflow.math.argmax(logits, axis=1)[0]


print(greedy("I went to the", 10))

## Test on some inputs

In [11]:
if lang == "fr":
    inputs = ["Je suis allé au",
              "Je me sens",
              "Comment vas-tu",
              "Comment se fait-il que",
              "Ça va merci, tu devrais",
              "Donald Trump vient de tweeter :",
              "Le président Trump vient de démissionner !"]
else:
    inputs = ["I went to the",
              "I am feeling",
              "How do you",
              "How comes that",
              "I'm fine thank you, you should",
              "Donald Trump just tweeted :",
              "President Trump just quit !"]

In [ ]:
def test_decoding(function, *args, **kwargs) -> None:
  generations = []
  for i in tqdm.notebook.tqdm(inputs, desc="Prompts", leave=False):
    generations.append(function(i, *args, **kwargs))
  for prompt, generation in zip(inputs, generations):
    print("—" * 80)
    print(f"Prompt: {prompt}")
    print(f"Generation: {generation}")
  print("—" * 80)

test_decoding(greedy, 50)

We can see that the answers "loop" quite quickly. It's a very classic flaw of greedy decoding.

## Sampling

As we've just seen, the results of greedy decoding have many problems. They loop quickly and are often very generic. The methods that follow aim to overcome these shortcomings.

The first approach we'll see consists in sampling from the model's results rather than always picking the most likely result.

In [ ]:
@decoding_loop
def sampling(logits: tensorflow.Tensor) -> tensorflow.Tensor:
  softmaxed = tensorflow.nn.log_softmax(logits)
  return tensorflow.random.categorical(softmaxed, 1)[0][0]

test_decoding(sampling, 50)

## Top-k Sampling

A variation of sampling consists in sampling only from the `k` highest results, to avoid generating a word that's too unlikely (even if that happens only rarely).

In [ ]:
@decoding_loop
def k_sampling(logits: tensorflow.Tensor, k: int) -> tensorflow.Tensor:
  topk = tensorflow.math.top_k(logits, k=k)
  softmaxed = tensorflow.nn.log_softmax(topk.values)
  sampled = tensorflow.random.categorical(softmaxed, 1)[0][0]
  return tensorflow.cast(topk.indices[0, sampled], "int64")

test_decoding(k_sampling, 50, k=20)

## Top-p Sampling

Another improvement of sampling consists in considering only the first `n` outputs, whose summed probability exceeds `p`, and not the following ones.

In [ ]:
@decoding_loop
def k_p_sampling(logits: tensorflow.Tensor, k: int, p: float
                ) -> tensorflow.Tensor:
    topk = tensorflow.math.top_k(logits, k=k)
    softmaxed = tensorflow.nn.softmax(topk.values)
    current_proba_sum = 0
    j = 0
    while current_proba_sum < p:
      current_proba_sum += softmaxed[0, j]
      j += 1
    sampled = tensorflow.random.categorical(
        tensorflow.math.log(softmaxed[:, :j]), 1)[0][0]
    return tensorflow.cast(topk.indices[0, sampled], "int64")

test_decoding(k_p_sampling, length=100, k=20, p=0.85)

## Using the `transformers` Functions

In the future, if you need to generate text, you can use the `generate` function of the `transformers` library's models. Here's an example:

In [ ]:
inputs = ["The NASA just tweeted:"]

In [18]:
def transformers_generate(prompt: str,
                          length: int,
                          temperature: float,
                          top_k: int,
                          top_p: float) -> str:
  token_ids = encode(prompt)
  generated = model.generate(input_ids=tensorflow.cast(token_ids, "int32"),
                             max_length=len(token_ids) + length,
                             temperature=temperature,
                             top_k=top_k,
                             top_p=top_p,
                             do_sample=True,
                             num_return_sequences=1)
  return decode(tensorflow.cast(generated, "int64"))

test_decoding(transformers_generate, length=1000, temperature=1, top_k=30, top_p=0.97)

Prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Setting `pad_token_id` to 50256 (first `eos_token_id`) to generate sequence


————————————————————————————————————————————————————————————————————————————————
Prompt: Kim jong Un resigns
Generation: Kim jong Un resigns as head of the country's ruling Communist party. Photo credit: Xinhua News Agency

SEOUL, March 6 (Xinhua) -- Kim Jong Un's political party is under pressure to step down, after a "very bitter experience" with a high-ranking member of the ruling Communist Party who was expelled from the party.

A parliamentary group has summoned Kim's two deputy vice-presidents, Jang Song Thaek and Park Geun-hye, and the party's vice-presidents, Hwang Kyo-ahn and Moon Jae-in, for questioning after reports of Kim's resignation last October.

The members of the parliamentary committee on the constitution said the former vice president was expelled because he tried to influence the party's leadership by taking part in anti-Korean protests during South Korean Independence Day in December.

The vice-presidents also accused Choi Soon-sil of attempting to take over the p